pré-processamento e geração dos dados

In [11]:
%pip install nltk spacy scikit-learn plotly seaborn matplotlib pandas numpy langdetect --quiet


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import csv
import json
import re
import sys
import os
from pathlib import Path

from langdetect import detect

import nltk
from nltk.stem import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from spacy.lang.pt.stop_words import STOP_WORDS as SPACY_PT_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer

from tqdm.auto import tqdm

nltk.download('punkt_tab', quiet=True)

print('dependencias carregadas.')

Dependencias carregadas.


definição das expressões regulares e ferramentas nlp

In [13]:
TOKENIZADOR_NLTK = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)
RADICALIZADOR_NLTK = SnowballStemmer("portuguese")

REGEX_TOKEN_COM_CONTEUDO = re.compile(r"\w", re.UNICODE)
REGEX_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
REGEX_MENCAO = re.compile(r"@\w+", re.UNICODE)
REGEX_HASHTAG = re.compile(r"#(\w+)", re.UNICODE)
REGEX_NAO_ALFANUMERICO = re.compile(r"[^\w\s]", re.UNICODE)
REGEX_ESPACOS = re.compile(r"\s+", re.UNICODE)
TOKENS_NUMERICOS = re.compile(r"^\d+$", re.UNICODE)

TEXTO_COM_EMOJIS = re.compile(
    r"[^a-zA-ZÀ-ÿ0-9\s.,!?'\-"
    r"#@"
    r"\U0001F300-\U0001F5FF"
    r"\U0001F600-\U0001F64F"
    r"\U0001F680-\U0001F6FF"
    r"\U0001F700-\U0001F77F"
    r"\U0001F780-\U0001F7FF"
    r"\U0001F800-\U0001F8FF"
    r"\U0001F900-\U0001F9FF"
    r"\U0001FA00-\U0001FAFF"
    r"\U00002700-\U000027BF"
    r"\U00002600-\U000026FF"
    r"\u200d\uFE0F]"
)

print('regex e ferramentas nlp definidas.')

Regex e ferramentas NLP definidas.


funções de pré-processamento

In [14]:
def normalizar_texto_csv(valor):
    if valor is None:
        return ""
    return str(valor)


def remover_decoracoes_com_regex(texto):
    return TEXTO_COM_EMOJIS.sub(" ", texto)


def remover_numericos_com_regex(texto):
    return " ".join(
        token
        for token in texto.split()
        if not TOKENS_NUMERICOS.match(token)
    )


def tokenizar_com_nltk(texto):
    return [
        token
        for token in TOKENIZADOR_NLTK.tokenize(texto)
        if REGEX_TOKEN_COM_CONTEUDO.search(token)
    ]


def remover_stopwords_com_spacy(tokens):
    return [
        token
        for token in tokens
        if token.casefold() not in SPACY_PT_STOP_WORDS
    ]


def aplicar_stemming_com_nltk(tokens):
    return " ".join(RADICALIZADOR_NLTK.stem(token.casefold()) for token in tokens)


def normalizar_com_regex(texto):
    texto_normalizado = texto.casefold()
    texto_normalizado = REGEX_URL.sub(" ", texto_normalizado)
    texto_normalizado = REGEX_HASHTAG.sub(r" \1 ", texto_normalizado)
    texto_normalizado = REGEX_MENCAO.sub(" ", texto_normalizado)
    texto_normalizado = REGEX_NAO_ALFANUMERICO.sub(" ", texto_normalizado)
    return REGEX_ESPACOS.sub(" ", texto_normalizado).strip()


def para_json(valor, ordenar_chaves=False):
    return json.dumps(valor, ensure_ascii=False, sort_keys=ordenar_chaves)

def detectar_idioma(texto):
    try:
        return detect(texto)
    except:
        return "unknown"


def processar_linha_preprocessamento(linha):
    texto = normalizar_texto_csv(linha.get("text"))

    idioma = detectar_idioma(texto)
    if idioma not in ("pt", "en"):
        return None
    linha["idioma_detectado"] = idioma

    texto = REGEX_URL.sub(" ", texto)

    texto = REGEX_MENCAO.sub(" ", texto)

    if not texto.strip():
        return None

    texto = remover_decoracoes_com_regex(texto)

    if not texto.strip():
        return None

    texto = remover_numericos_com_regex(texto)

    tokens = tokenizar_com_nltk(texto)

    tokens_sem_stopwords = remover_stopwords_com_spacy(tokens)

    texto_radicalizado = aplicar_stemming_com_nltk(tokens_sem_stopwords)

    texto_normalizado = normalizar_com_regex(texto_radicalizado)

    linha["tokenizacao_nltk"] = para_json(tokens)
    linha["remocao_stopwords_spacy"] = para_json(tokens_sem_stopwords)
    linha["stemming_nltk"] = texto_radicalizado
    linha["normalizacao_re"] = texto_normalizado

    return texto_normalizado

print('funcoes de pre-processamento definidas.')

Funcoes de pre-processamento definidas.


carregamento dos tweets brutos

In [15]:
CAMINHO_ENTRADA = Path('..') / 'data' / 'tweets.csv'

if not CAMINHO_ENTRADA.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {CAMINHO_ENTRADA.resolve()}')

with CAMINHO_ENTRADA.open('r', newline='', encoding='utf-8') as f:
    leitor = csv.DictReader(f)
    linhas = list(leitor)
    cabecalhos = list(leitor.fieldnames or [])

print(f'Tweets carregados: {len(linhas)}')
print(f'Colunas originais: {len(cabecalhos)}')
print(f'Amostra (texto): {linhas[0].get("text", "")[:120]}...')

Tweets carregados: 2815
Colunas originais: 28
Amostra (texto): Refeições de hj! #wieiad ₍ᐢ. .ᐢ₎
．☆．。．:*･ﾟ

Gente eu esqueci de tirar foto do almoço, mas foi arroz com cenouras e um st...


execução da pipeline de pré-processamento

In [16]:
total_original = len(linhas)
textos_normalizados = []
linhas_validas = []

for linha in tqdm(linhas, desc='Pre-processando tweets'):
    texto_norm = processar_linha_preprocessamento(linha)
    if texto_norm is None:
        continue
    textos_normalizados.append(texto_norm)
    linhas_validas.append(linha)

linhas = linhas_validas

print(f'\nTweets pre-processados: {len(textos_normalizados)}')
print(f'Tweets descartados: {total_original - len(textos_normalizados)}')

print('\nexemplo:')
print(f'  Original:      {linhas[0].get("text", "")[:120]}')
print(f'  Tokens (NLTK): {linhas[0].get("tokenizacao_nltk", "")[:120]}')
print(f'  Stemmed:       {linhas[0].get("stemming_nltk", "")[:120]}')
print(f'  normalizacao_re: {linhas[0].get("normalizacao_re", "")[:120]}')

Pre-processando tweets: 100%|██████████| 2815/2815 [00:16<00:00, 172.32it/s]


Tweets pre-processados: 2353
Tweets descartados: 462

Exemplo:
  Original:      Refeições de hj! #wieiad ₍ᐢ. .ᐢ₎
．☆．。．:*･ﾟ

Gente eu esqueci de tirar foto do almoço, mas foi arroz com cenouras e um st
  Tokens (NLTK): ["Refeições", "de", "hj", "#wieiad", "Gente", "eu", "esqueci", "de", "tirar", "foto", "do", "almoço", "mas", "foi", "arr
  Stemmed:       refeiçõ hj #wieiad gent esquec tir fot almoc arroz cenour steak frang 1.244 kcals 43.2 g prot gast klcals caminh muscul 
  normalizacao_re: refeiçõ hj wieiad gent esquec tir fot almoc arroz cenour steak frang 1 244 kcals 43 2 g prot gast klcals caminh muscul m


tf-idf com scikit-learn

In [17]:
vetorizador = TfidfVectorizer(
    lowercase=False,
    max_df=0.85,
    min_df=2,
    max_features=1000,
    token_pattern=r"(?u)\b\w\w+\b",
)

textos_para_vetorizar = [t if t else ' ' for t in textos_normalizados]

if any(textos_para_vetorizar):
    matriz = vetorizador.fit_transform(textos_para_vetorizar)
    nomes_features = vetorizador.get_feature_names_out().tolist()
else:
    matriz = None
    nomes_features = []

print(f'Features TF-IDF: {len(nomes_features)} termos')
print(f'Matriz TF-IDF: {matriz.shape if matriz is not None else "vazia"}')
print(f'\nTermos mais frequentes (soma TF-IDF):')
if matriz is not None:
    import numpy as np
    soma_tfidf = np.array(matriz.sum(axis=0)).flatten()
    top_idx = np.argsort(soma_tfidf)[-15:][::-1]
    for idx in top_idx:
        print(f'  {nomes_features[idx]:20s} {soma_tfidf[idx]:.2f}')

Features TF-IDF: 1000 termos
Matriz TF-IDF: (2353, 1000)

Termos mais frequentes (soma TF-IDF):
  com                  81.31
  pra                  68.92
  vou                  50.40
  thread               48.33
  almoc                45.90
  recovery             43.40
  hoj                  42.21
  pro                  39.12
  fic                  38.42
  to                   36.74
  ed                   35.04
  calor                34.48
  dia                  33.18
  nao                  32.68
  emagrec              32.10


In [18]:
features_por_linha = [para_json({}) for _ in linhas]

if matriz is not None:
    for indice, vetor_linha in enumerate(tqdm(matriz, desc='Extraindo features TF-IDF'), start=1):
        mapa_features = {
            nomes_features[indice_coluna]: round(float(valor), 6)
            for indice_coluna, valor in zip(vetor_linha.indices, vetor_linha.data)
        }
        features_por_linha[indice - 1] = para_json(mapa_features, ordenar_chaves=True)

for linha, feat_json in zip(linhas, features_por_linha):
    linha["features_tfidf_sklearn"] = feat_json

print('features tf-idf salvas nas linhas.')

Extraindo features TF-IDF: 2353it [00:00, 43674.23it/s]

Features TF-IDF salvas nas linhas.


exportação: `entrega_2.csv` e `entrega_2_tfidf_features.json`

In [19]:
PASTA_SAIDA = Path('..') / 'entregas' / 'p2'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

caminho_csv = PASTA_SAIDA / 'entrega_2.csv'
caminho_json = PASTA_SAIDA / 'entrega_2_tfidf_features.json'

PASTA_ENTREGA_1 = Path('..') / 'entregas' / 'p1'
PASTA_ENTREGA_1.mkdir(parents=True, exist_ok=True)
caminho_csv_p1 = PASTA_ENTREGA_1 / 'entrega_1.csv'

In [20]:
novas_colunas = [
    "idioma_detectado",
    "tokenizacao_nltk",
    "remocao_stopwords_spacy",
    "stemming_nltk",
    "normalizacao_re",
    "features_tfidf_sklearn",
]

cabecalhos_saida = [*cabecalhos, *novas_colunas]

with caminho_csv.open('w', newline='', encoding='utf-8') as f:
    escritor = csv.DictWriter(f, fieldnames=cabecalhos_saida)
    escritor.writeheader()
    escritor.writerows(linhas)

print(f'entrega_2.csv salvo: {caminho_csv.resolve()}')
print(f'  Linhas: {len(linhas)}')
print(f'  Colunas: {len(cabecalhos_saida)}')

entrega_2.csv salvo: /Users/pedrohenriquewindisch/projects/edtwt/entregas/p2/entrega_2.csv
  Linhas: 2353
  Colunas: 34


In [21]:
params = vetorizador.get_params()
metadados = {
    "feature_names": nomes_features,
    "vectorizer": {
        "max_df": params["max_df"],
        "min_df": params["min_df"],
        "max_features": params["max_features"],
        "token_pattern": params["token_pattern"],
        "lowercase": params["lowercase"],
    },
}

caminho_json.write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(f'entrega_2_tfidf_features.json salvo: {caminho_json.resolve()}')
print(f'  Features: {len(nomes_features)}')

entrega_2_tfidf_features.json salvo: /Users/pedrohenriquewindisch/projects/edtwt/entregas/p2/entrega_2_tfidf_features.json
  Features: 1000


In [22]:
colunas_excluidas_p1 = {"idioma_detectado", "normalizacao_re", "features_tfidf_sklearn"}
colunas_p1 = [c for c in cabecalhos_saida if c not in colunas_excluidas_p1]
linhas_p1 = [{k: v for k, v in l.items() if k not in colunas_excluidas_p1} for l in linhas]
with caminho_csv_p1.open('w', newline='', encoding='utf-8') as f:
    escritor = csv.DictWriter(f, fieldnames=colunas_p1)
    escritor.writeheader()
    escritor.writerows(linhas_p1)

print(f'entrega_1.csv salvo: {caminho_csv_p1.resolve()}')
print(f'  Linhas: {len(linhas)}')
print(f'  Colunas: {len(colunas_p1)}')

entrega_1.csv salvo: /Users/pedrohenriquewindisch/projects/edtwt/entregas/p1/entrega_1.csv
  Linhas: 2353
  Colunas: 31


verificação: termos mais frequentes

In [23]:
import numpy as np

print('Top 15 termos por soma TF-IDF (nao devem conter https, co, t, etc.):')
soma_tfidf = np.array(matriz.sum(axis=0)).flatten()
top_idx = np.argsort(soma_tfidf)[-15:][::-1]
for idx in top_idx:
    print(f'  {nomes_features[idx]:20s} {soma_tfidf[idx]:.2f}')

termos_suspeitos = ['https', 'co', 'www', 'http', 'com', 't']
presentes = [t for t in termos_suspeitos if t in nomes_features]
if presentes:
    print(f'\nATENCAO: Termos suspeitos ainda presentes no vocabulario: {presentes}')
else:
    print('\nok: nenhum termo suspeito de fragmento de url encontrado.')

Top 15 termos por soma TF-IDF (nao devem conter https, co, t, etc.):
  com                  81.31
  pra                  68.92
  vou                  50.40
  thread               48.33
  almoc                45.90
  recovery             43.40
  hoj                  42.21
  pro                  39.12
  fic                  38.42
  to                   36.74
  ed                   35.04
  calor                34.48
  dia                  33.18
  nao                  32.68
  emagrec              32.10

ATENCAO: Termos suspeitos ainda presentes no vocabulario: ['com']


próximos passos